# 5-Class Intent Classification — Leakage-Safe Manual 5-CV Training

This notebook **does not create or reshuffle cross-validation folds**. It consumes the frozen 5-class CV split files already produced by the manual fold-building notebook.

Methodology is intentionally kept the same as the earlier 4-class CV training notebook:

- frozen conversation-level Train / Validation / Test memberships
- complete dialogue history used to classify each target user turn
- fold-specific LLM augmentation added to **training only**
- independent translated Chinese→English corpus added to **training only**
- balanced class weights computed from each fold's **final training data only**
- validation Macro-F1 selects the best in-memory model state
- outer held-out test data is evaluated only after the best validation state is restored
- BERT-base uses the same hyperparameters: LR `2e-5`, weight decay `0.01`, batch size `8`, max length `256`, 5 epochs

The only task-level changes are:
1. the target label set now includes `Inquire`,
2. the frozen experiment contains 826 eligible conversations / 3,196 target turns,
3. CV1 has a 166-conversation test fold, CV2–CV5 have 165,
4. LLM augmentation contains Chitchat + Revise + fold-specific Inquire rows.


In [1]:
# Cell 1 — Install dependencies
!pip install -q "transformers>=4.46,<5" datasets accelerate sentencepiece scikit-learn sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 3 — Imports
import os
import re
import gc
import glob
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    set_seed,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
import matplotlib.pyplot as plt


In [4]:
# Cell 4 — Master experiment configuration
# Same structure as the 4-class notebook; updated only for the 5-class Drive layout.

BASE_DIR = '/content/drive/MyDrive/5cv_5class_intent_12Aug'

SOURCE_PATH = os.path.join(BASE_DIR, 'source', 'conversation_en.json')
TRANSLATED_PATH = os.path.join(
    BASE_DIR,
    'translated_data',
    'conversation_cn_translated_en_deepseek_v4flash.json',
)
SPLIT_DIR = os.path.join(BASE_DIR, '5class_splits')
LLM_AUG_DIR = os.path.join(BASE_DIR, 'llm_augmentation')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

VALID_LABELS = ['Chitchat', 'Interpret', 'Reveal', 'Revise', 'Inquire']
label2id = {label: i for i, label in enumerate(VALID_LABELS)}
id2label = {i: label for label, i in label2id.items()}

# Current final fold-specific Inquire augmentation counts.
EXPECTED_INQUIRE = {
    1: 27,
    2: 25,
    3: 30,
    4: 32,
    5: 35,
}

RANDOM_STATE = 42
MAX_LEN = 256
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

# SAME optimization hyperparameters as the 4-class experiment.
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

RUN_FOLDS = [1, 2, 3, 4, 5]

MODEL_CONFIGS = {
    'BERT-base': {
        'model_name': 'bert-base-uncased',
        'enabled': False,
    },
    'DistilBERT': {
        'model_name': 'distilbert-base-uncased',
        'enabled': True,
    },
    'RoBERTa': {
        'model_name': 'roberta-base',
        'enabled': True,
    },
    'MiniLM': {
        'model_name': 'microsoft/MiniLM-L12-H384-uncased',
        'enabled': True,
    },
}

print('BASE_DIR:', BASE_DIR)
print('RUN_FOLDS:', RUN_FOLDS)
print('Enabled models:', [k for k,v in MODEL_CONFIGS.items() if v['enabled']])
print('Labels:', VALID_LABELS)
print('Shared LR:', LEARNING_RATE, '| Shared weight decay:', WEIGHT_DECAY)


BASE_DIR: /content/drive/MyDrive/5cv_5class_intent_12Aug
RUN_FOLDS: [1, 2, 3, 4, 5]
Enabled models: ['DistilBERT', 'RoBERTa', 'MiniLM']
Labels: ['Chitchat', 'Interpret', 'Reveal', 'Revise', 'Inquire']
Shared LR: 2e-05 | Shared weight decay: 0.01


In [5]:
# Cell 5 — Verify expected files exist

required_paths = [SOURCE_PATH, TRANSLATED_PATH]

for cv in range(1, 6):
    required_paths.extend([
        os.path.join(SPLIT_DIR, f'cv{cv}_train_ids.csv'),
        os.path.join(SPLIT_DIR, f'cv{cv}_val_ids.csv'),
        os.path.join(SPLIT_DIR, f'cv{cv}_test_ids.csv'),
    ])

missing = [p for p in required_paths if not os.path.exists(p)]

if missing:
    print('Missing required files:')
    for p in missing:
        print('  ', p)
    raise FileNotFoundError(
        'One or more required files are missing. '
        'If the source/translated JSON filenames differ, update SOURCE_PATH '
        'or TRANSLATED_PATH in Cell 4.'
    )

# Verify exactly one final combined LLM augmentation file exists per CV.
for cv in range(1, 6):
    matches = sorted(glob.glob(os.path.join(
        LLM_AUG_DIR,
        f'cv{cv}_synthetic_100_chitchat_100_revise_*_inquire.csv'
    )))
    assert len(matches) == 1, (
        f'CV{cv}: expected exactly one final combined augmentation file; '
        f'found {len(matches)}: {matches}'
    )

print('PASS: source, translated data, all frozen split files, and all final LLM augmentation files exist.')


PASS: source, translated data, all frozen split files, and all final LLM augmentation files exist.


## Data loading and context construction

The original English corpus keeps **all dialogue turns** for history construction. All five target intents — `Chitchat`, `Interpret`, `Reveal`, `Revise`, and `Inquire` — are prediction targets.


In [6]:
# Cell 6 — Generic conversation JSON parser

def conversations_to_turn_df(raw_conversations, id_prefix=''):
    rows = []
    for conv in raw_conversations:
        cid = id_prefix + str(conv['conv_id'])
        for turn_index, turn in enumerate(conv.get('conversation', [])):
            rows.append({
                'conversation_id': cid,
                'turn_index': int(turn_index),
                'role': str(turn.get('role', '')).lower().strip(),
                'text': turn.get('content', '') or '',
                'intent': turn.get('intent'),
            })
    return (
        pd.DataFrame(rows)
        .sort_values(['conversation_id', 'turn_index'])
        .reset_index(drop=True)
    )


def build_turn_lookup(turn_df):
    return {
        str(cid): g.sort_values('turn_index').reset_index(drop=True)
        for cid, g in turn_df.groupby('conversation_id')
    }


def build_context_dataset(target_rows, turns_by_conv, return_meta=False):
    texts, labels, meta = [], [], []

    for _, row in target_rows.iterrows():
        cid = str(row['conversation_id'])
        ti = int(row['turn_index'])
        assert cid in turns_by_conv, f'Missing history for conversation {cid}'

        conv_turns = turns_by_conv[cid]
        previous = conv_turns[conv_turns['turn_index'] < ti]

        history = []
        for _, p in previous.iterrows():
            txt = str(p['text']).strip()
            if not txt:
                continue
            history.append(f"{str(p['role']).upper()}: {txt}")

        current = str(row['text']).strip()
        full_text = ('\n'.join(history) + '\nUSER: ' + current) if history else ('USER: ' + current)

        texts.append(full_text)
        labels.append(str(row['intent']))
        meta.append((cid, ti))

    if return_meta:
        return np.array(texts), np.array(labels), meta
    return np.array(texts), np.array(labels)

In [7]:
# Cell 7 — Load original English corpus
with open(SOURCE_PATH, 'r', encoding='utf-8') as f:
    raw_en = json.load(f)

en_turns_full = conversations_to_turn_df(raw_en)
en_turns_by_conv = build_turn_lookup(en_turns_full)

en_target_df = en_turns_full[
    (en_turns_full['role'] == 'user')
    & (en_turns_full['intent'].isin(VALID_LABELS))
].copy().reset_index(drop=True)

print('Original English conversations:', en_turns_full['conversation_id'].nunique())
print('5-class eligible English conversations:', en_target_df['conversation_id'].nunique())
print('5-class English target turns:', len(en_target_df))
print('\nEnglish target distribution:')
print(en_target_df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0))

assert en_target_df['conversation_id'].nunique() == 826
assert len(en_target_df) == 3196

expected_class_totals = {
    'Chitchat': 268,
    'Interpret': 1813,
    'Reveal': 1031,
    'Revise': 67,
    'Inquire': 17,
}

actual_class_totals = en_target_df['intent'].value_counts().to_dict()

for label, expected in expected_class_totals.items():
    assert int(actual_class_totals.get(label, 0)) == expected, (
        f'{label}: expected {expected}, found {actual_class_totals.get(label, 0)}'
    )

print('\nPASS: 5-class English corpus matches 826 eligible conversations / 3,196 target turns.')


Original English conversations: 826
5-class eligible English conversations: 826
5-class English target turns: 3196

English target distribution:
intent
Chitchat      268
Interpret    1813
Reveal       1031
Revise         67
Inquire        17
Name: count, dtype: int64

PASS: 5-class English corpus matches 826 eligible conversations / 3,196 target turns.


In [8]:
# Cell 8 — Load independent translated Chinese→English corpus, namespace IDs
with open(TRANSLATED_PATH, 'r', encoding='utf-8') as f:
    raw_cn = json.load(f)

# Capture raw numeric/string IDs only for diagnostics; they are NOT used for joining.
raw_cn_ids = {str(conv['conv_id']) for conv in raw_cn}
raw_en_ids = {str(conv['conv_id']) for conv in raw_en}
raw_numeric_id_overlap = raw_cn_ids & raw_en_ids
print('Raw English/CN numeric ID overlap before namespacing:', len(raw_numeric_id_overlap))
print('This is not treated as leakage because the translated corpus is independent; CN IDs are namespaced below.')

cn_turns_full = conversations_to_turn_df(raw_cn, id_prefix='CN_')
cn_turns_by_conv = build_turn_lookup(cn_turns_full)
cn_target_df = cn_turns_full[
    (cn_turns_full['role'] == 'user')
    & (cn_turns_full['intent'].isin(VALID_LABELS))
].copy().reset_index(drop=True)

assert cn_turns_full['conversation_id'].astype(str).str.startswith('CN_').all()
assert cn_target_df['conversation_id'].astype(str).str.startswith('CN_').all()

X_cn_train, y_cn_train, cn_meta = build_context_dataset(
    cn_target_df, cn_turns_by_conv, return_meta=True
)

print('\nTranslated CN→EN target turns:', len(cn_target_df))
print('Translated label distribution:')
print(cn_target_df['intent'].value_counts().reindex(VALID_LABELS, fill_value=0))

Raw English/CN numeric ID overlap before namespacing: 6
This is not treated as leakage because the translated corpus is independent; CN IDs are namespaced below.

Translated CN→EN target turns: 3056
Translated label distribution:
intent
Chitchat      275
Interpret    1590
Reveal        975
Revise        215
Inquire         1
Name: count, dtype: int64


## Frozen split and augmentation loaders

No random splitting occurs below. All membership comes from the frozen CSV files.

In [9]:
# Cell 9 — Load one frozen CV split

def _read_id_file(path):
    df = pd.read_csv(path)
    if 'conversation_id' in df.columns:
        col = 'conversation_id'
    elif 'conv_id' in df.columns:
        col = 'conv_id'
    else:
        raise ValueError(
            f'{path} must contain conversation_id or conv_id. '
            f'Found columns: {list(df.columns)}'
        )
    return set(df[col].astype(str))


def load_cv_ids(cv_run):
    train_ids = _read_id_file(
        os.path.join(SPLIT_DIR, f'cv{cv_run}_train_ids.csv')
    )
    val_ids = _read_id_file(
        os.path.join(SPLIT_DIR, f'cv{cv_run}_val_ids.csv')
    )
    test_ids = _read_id_file(
        os.path.join(SPLIT_DIR, f'cv{cv_run}_test_ids.csv')
    )
    return train_ids, val_ids, test_ids


def find_llm_aug_file(cv_run):
    matches = sorted(glob.glob(os.path.join(
        LLM_AUG_DIR,
        f'cv{cv_run}_synthetic_100_chitchat_100_revise_*_inquire.csv'
    )))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one final CV{cv_run} LLM augmentation file; '
            f'found {len(matches)}: {matches}'
        )
    return matches[0]


def load_llm_augmentation(cv_run):
    path = find_llm_aug_file(cv_run)
    df = pd.read_csv(path)
    df['conversation_id'] = df['conversation_id'].astype(str)
    df['turn_index'] = df['turn_index'].astype(int)
    df['intent'] = df['intent'].astype(str).str.strip()
    df['text'] = df['text'].fillna('').astype(str)
    return df, path


In [10]:
# Cell 10 — Deep leakage audit for a single 5-class CV fold

def audit_cv_fold(cv_run, verbose=True):
    train_ids, val_ids, test_ids = load_cv_ids(cv_run)
    aug_df, aug_path = load_llm_augmentation(cv_run)

    expected_train = 528 if cv_run == 1 else 529
    expected_val = 132
    expected_test = 166 if cv_run == 1 else 165

    # 1) Frozen conversation memberships
    assert len(train_ids) == expected_train, (
        f'CV{cv_run}: train conversation count != {expected_train}'
    )
    assert len(val_ids) == expected_val, (
        f'CV{cv_run}: val conversation count != {expected_val}'
    )
    assert len(test_ids) == expected_test, (
        f'CV{cv_run}: test conversation count != {expected_test}'
    )

    assert train_ids.isdisjoint(val_ids), f'CV{cv_run}: train/val conversation leakage'
    assert train_ids.isdisjoint(test_ids), f'CV{cv_run}: train/test conversation leakage'
    assert val_ids.isdisjoint(test_ids), f'CV{cv_run}: val/test conversation leakage'

    assert len(train_ids | val_ids | test_ids) == 826, (
        f'CV{cv_run}: split union != 826'
    )

    # 2) All IDs must exactly match the 826 eligible 5-class English conversations.
    eligible_ids = set(en_target_df['conversation_id'].astype(str))
    assert (train_ids | val_ids | test_ids) == eligible_ids, (
        f'CV{cv_run}: split IDs do not exactly match eligible English IDs'
    )

    # 3) Reconstruct English target rows and verify no target-key overlap.
    train_df = en_target_df[
        en_target_df['conversation_id'].isin(train_ids)
    ].copy()

    val_df = en_target_df[
        en_target_df['conversation_id'].isin(val_ids)
    ].copy()

    test_df = en_target_df[
        en_target_df['conversation_id'].isin(test_ids)
    ].copy()

    key = lambda d: set(
        zip(
            d['conversation_id'].astype(str),
            d['turn_index'].astype(int)
        )
    )

    train_keys, val_keys, test_keys = key(train_df), key(val_df), key(test_df)

    assert train_keys.isdisjoint(val_keys), f'CV{cv_run}: target-turn train/val leakage'
    assert train_keys.isdisjoint(test_keys), f'CV{cv_run}: target-turn train/test leakage'
    assert val_keys.isdisjoint(test_keys), f'CV{cv_run}: target-turn val/test leakage'

    # 4) LLM augmentation structure and fold-specific counts.
    required_aug_cols = {
        'conversation_id', 'turn_index', 'role', 'text', 'intent',
        'is_augmented', 'source_user_text'
    }

    missing_cols = required_aug_cols - set(aug_df.columns)
    assert not missing_cols, f'CV{cv_run}: missing LLM columns: {missing_cols}'

    expected_inquire = EXPECTED_INQUIRE[cv_run]
    expected_llm_total = 200 + expected_inquire

    assert len(aug_df) == expected_llm_total, (
        f'CV{cv_run}: expected {expected_llm_total} synthetic rows, '
        f'found {len(aug_df)}'
    )

    aug_counts = aug_df['intent'].value_counts()

    assert int(aug_counts.get('Chitchat', 0)) == 100, (
        f'CV{cv_run}: Chitchat synthetic count != 100'
    )
    assert int(aug_counts.get('Revise', 0)) == 100, (
        f'CV{cv_run}: Revise synthetic count != 100'
    )
    assert int(aug_counts.get('Inquire', 0)) == expected_inquire, (
        f'CV{cv_run}: Inquire synthetic count != {expected_inquire}'
    )

    assert set(aug_df['intent'].dropna()) <= {
        'Chitchat', 'Revise', 'Inquire'
    }

    # 5) LLM source conversations must be TRAIN only.
    aug_ids = set(aug_df['conversation_id'].astype(str))

    assert aug_ids.issubset(train_ids), f'CV{cv_run}: LLM source outside frozen train'
    assert aug_ids.isdisjoint(val_ids), f'CV{cv_run}: LLM source overlaps validation'
    assert aug_ids.isdisjoint(test_ids), f'CV{cv_run}: LLM source overlaps test'

    if 'is_augmented' in aug_df.columns:
        aug_flag = (
            aug_df['is_augmented']
            .astype(str)
            .str.lower()
            .isin(['true', '1', '1.0'])
        )
        assert aug_flag.all(), (
            f'CV{cv_run}: non-augmented row found in synthetic file'
        )

    # Same source-specific exact duplicate identity check used in the original methodology.
    dup_syn = aug_df.duplicated(
        ['conversation_id', 'turn_index', 'text', 'intent']
    ).sum()

    assert dup_syn == 0, (
        f'CV{cv_run}: duplicate synthetic rows found'
    )

    # Additional post-generation global exact-text audit.
    # This is reported but not treated as conversation leakage.
    normalized_aug_text = (
        aug_df['text']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
    )

    global_exact_text_duplicates = int(
        normalized_aug_text.duplicated().sum()
    )

    # 6) Every synthetic row must map to a genuine original TRAIN target
    #    turn with the SAME source intent, including Inquire.
    original_train_map = {
        (str(r.conversation_id), int(r.turn_index)): str(r.intent)
        for r in train_df[
            ['conversation_id', 'turn_index', 'intent']
        ].itertuples(index=False)
    }

    bad_source = []
    bad_label = []

    for r in aug_df[
        ['conversation_id', 'turn_index', 'intent']
    ].itertuples(index=False):

        k = (
            str(r.conversation_id),
            int(r.turn_index)
        )

        if k not in original_train_map:
            bad_source.append(k)

        elif original_train_map[k] != str(r.intent):
            bad_label.append(
                (k, original_train_map[k], str(r.intent))
            )

    assert not bad_source, (
        f'CV{cv_run}: synthetic rows with missing original train source turn: '
        f'{bad_source[:5]}'
    )

    assert not bad_label, (
        f'CV{cv_run}: synthetic labels differ from original source label: '
        f'{bad_label[:5]}'
    )

    # 7) Independent translated corpus is structurally isolated by namespace.
    assert cn_target_df['conversation_id'].astype(str).str.startswith('CN_').all()
    assert not any(
        str(x).startswith('CN_')
        for x in train_ids | val_ids | test_ids
    )

    # Distribution summaries.
    def counts_dict(df):
        c = df['intent'].value_counts().reindex(
            VALID_LABELS,
            fill_value=0
        )
        return {
            label: int(c[label])
            for label in VALID_LABELS
        }

    train_counts = counts_dict(train_df)
    val_counts = counts_dict(val_df)
    test_counts = counts_dict(test_df)

    cn_counts = (
        cn_target_df['intent']
        .value_counts()
        .reindex(VALID_LABELS, fill_value=0)
    )

    aug_by_label = (
        aug_df['intent']
        .value_counts()
        .reindex(VALID_LABELS, fill_value=0)
    )

    final_train_counts = {
        label:
            train_counts[label]
            + int(aug_by_label[label])
            + int(cn_counts[label])
        for label in VALID_LABELS
    }

    row = {
        'CV': cv_run,
        'Train_conversations': len(train_ids),
        'Val_conversations': len(val_ids),
        'Test_conversations': len(test_ids),
        'Train_original_targets': len(train_df),
        'Val_targets': len(val_df),
        'Test_targets': len(test_df),
        'LLM_rows': len(aug_df),
        'LLM_source_conversations': len(aug_ids),
        'LLM_global_exact_text_duplicates': global_exact_text_duplicates,
        'Translated_targets': len(cn_target_df),
        'Final_train_examples':
            len(train_df) + len(aug_df) + len(cn_target_df),
        'Augmentation_file': os.path.basename(aug_path),
    }

    for label in VALID_LABELS:
        row[f'TrainOriginal_{label}'] = train_counts[label]
        row[f'Val_{label}'] = val_counts[label]
        row[f'Test_{label}'] = test_counts[label]
        row[f'LLM_{label}'] = int(aug_by_label[label])
        row[f'Translated_{label}'] = int(cn_counts[label])
        row[f'FinalTrain_{label}'] = final_train_counts[label]

    if verbose:
        print(
            f'CV{cv_run}: PASS — no conversation-ID or target-turn leakage; '
            'LLM provenance verified.'
        )
        print(
            '  Train/Val/Test conversations:',
            len(train_ids), len(val_ids), len(test_ids)
        )
        print(
            '  Original target rows:',
            len(train_df), len(val_df), len(test_df)
        )
        print(
            '  LLM distribution:',
            aug_df['intent'].value_counts().to_dict()
        )
        print(
            '  LLM rows / source conversations:',
            len(aug_df), len(aug_ids)
        )
        print(
            '  LLM global exact-text duplicates (audit only):',
            global_exact_text_duplicates
        )
        print('  LLM source ∩ Validation:', len(aug_ids & val_ids))
        print('  LLM source ∩ Test:', len(aug_ids & test_ids))

    return row


In [11]:
# Cell 11 — Audit ALL 5 CV folds before any training

fold_audit_rows = [
    audit_cv_fold(cv, verbose=True)
    for cv in range(1, 6)
]

fold_data_summary = pd.DataFrame(fold_audit_rows)

# Across outer CV: every eligible conversation must be TEST exactly once.
all_test_ids = []

for cv in range(1, 6):
    _, _, test_ids = load_cv_ids(cv)
    all_test_ids.extend(sorted(test_ids))

assert len(all_test_ids) == 826
assert len(set(all_test_ids)) == 826
assert set(all_test_ids) == set(
    en_target_df['conversation_id'].astype(str)
)

# Pairwise outer test folds must be disjoint.
for i in range(1, 6):
    ti = load_cv_ids(i)[2]

    for j in range(i + 1, 6):
        tj = load_cv_ids(j)[2]

        assert ti.isdisjoint(tj), (
            f'Outer test fold overlap between CV{i} and CV{j}'
        )

print(
    '\nPASS: all five outer test folds are pairwise disjoint '
    'and cover all 826 eligible conversations exactly once.'
)

summary_path = os.path.join(
    RESULTS_DIR,
    'fold_data_and_leakage_audit.csv'
)

fold_data_summary.to_csv(
    summary_path,
    index=False
)

print('Saved fold audit summary:', summary_path)

display(fold_data_summary)


CV1: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 166
  Original target rows: 2030 526 640
  LLM distribution: {'Chitchat': 100, 'Revise': 100, 'Inquire': 27}
  LLM rows / source conversations: 227 76
  LLM global exact-text duplicates (audit only): 0
  LLM source ∩ Validation: 0
  LLM source ∩ Test: 0
CV2: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 529 132 165
  Original target rows: 2046 511 639
  LLM distribution: {'Chitchat': 100, 'Revise': 100, 'Inquire': 25}
  LLM rows / source conversations: 225 69
  LLM global exact-text duplicates (audit only): 62
  LLM source ∩ Validation: 0
  LLM source ∩ Test: 0
CV3: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 529 132 165
  Original target rows: 2042 516 638
  LLM distribution: {'Chitchat': 100, 'Revise': 100, 'Inquire': 30}
  LLM rows / source co

,CV,Train_conversations,Val_conversations,Test_conversations,Train_original_targets,Val_targets,Test_targets,LLM_rows,LLM_source_conversations,LLM_global_exact_text_duplicates,Translated_targets,Final_train_examples,Augmentation_file,TrainOriginal_Chitchat,Val_Chitchat,Test_Chitchat,LLM_Chitchat,Translated_Chitchat,FinalTrain_Chitchat,TrainOriginal_Interpret,Val_Interpret,Test_Interpret,LLM_Interpret,Translated_Interpret,FinalTrain_Interpret,TrainOriginal_Reveal,Val_Reveal,Test_Reveal,LLM_Reveal,Translated_Reveal,FinalTrain_Reveal,TrainOriginal_Revise,Val_Revise,Test_Revise,LLM_Revise,Translated_Revise,FinalTrain_Revise,TrainOriginal_Inquire,Val_Inquire,Test_Inquire,LLM_Inquire,Translated_Inquire,FinalTrain_Inquire
0,1,528,132,166,2030,526,640,227,76,0,3056,5313,cv1_synthetic_100_chitchat_100_revise_27_inquire.csv,173,42,53,100,275,548,1147,303,363,0,1590,2737,660,164,207,0,975,1635,39,15,13,100,215,354,11,2,4,27,1,39
1,2,529,132,165,2046,511,639,225,69,62,3056,5327,cv2_synthetic_100_chitchat_100_revise_25_inquire.csv,174,40,54,100,275,549,1163,288,362,0,1590,2753,653,172,206,0,975,1628,46,8,13,100,215,361,10,3,4,25,1,36
2,3,529,132,165,2042,516,638,230,74,59,3056,5328,cv3_synthetic_100_chitchat_100_revise_30_inquire.csv,166,49,53,100,275,541,1145,306,362,0,1590,2735,674,151,206,0,975,1649,45,8,14,100,215,360,12,2,3,30,1,43
3,4,529,132,165,2063,494,639,232,71,52,3056,5351,cv4_synthetic_100_chitchat_100_revise_32_inquire.csv,177,37,54,100,275,552,1164,286,363,0,1590,2754,669,156,206,0,975,1644,40,14,13,100,215,355,13,1,3,32,1,46
4,5,529,132,165,2024,532,640,235,67,45,3056,5315,cv5_synthetic_100_chitchat_100_revise_35_inquire.csv,169,45,54,100,275,544,1134,316,363,0,1590,2724,658,167,206,0,975,1633,49,4,14,100,215,364,14,0,3,35,1,50


In [12]:
# Cell 12 — Compact original split distribution table for all five CV runs

rows = []

for cv in range(1, 6):
    train_ids, val_ids, test_ids = load_cv_ids(cv)

    for split_name, ids in [
        ('Train', train_ids),
        ('Validation', val_ids),
        ('Test', test_ids)
    ]:
        d = en_target_df[
            en_target_df['conversation_id'].isin(ids)
        ]

        c = d['intent'].value_counts().reindex(
            VALID_LABELS,
            fill_value=0
        )

        rows.append({
            'CV': f'CV{cv}',
            'Split': split_name,
            'Conversations': len(ids),
            'Chitchat': int(c['Chitchat']),
            'Interpret': int(c['Interpret']),
            'Reveal': int(c['Reveal']),
            'Revise': int(c['Revise']),
            'Inquire': int(c['Inquire']),
            'Total_examples': int(len(d)),
        })

original_split_distribution = pd.DataFrame(rows)

display(original_split_distribution)

original_split_distribution.to_csv(
    os.path.join(
        RESULTS_DIR,
        'original_cv_split_distribution.csv'
    ),
    index=False,
)


,CV,Split,Conversations,Chitchat,Interpret,Reveal,Revise,Inquire,Total_examples
0,CV1,Train,528,173,1147,660,39,11,2030
1,CV1,Validation,132,42,303,164,15,2,526
2,CV1,Test,166,53,363,207,13,4,640
3,CV2,Train,529,174,1163,653,46,10,2046
4,CV2,Validation,132,40,288,172,8,3,511
5,CV2,Test,165,54,362,206,13,4,639
6,CV3,Train,529,166,1145,674,45,12,2042
7,CV3,Validation,132,49,306,151,8,2,516
8,CV3,Test,165,53,362,206,14,3,638
9,CV4,Train,529,177,1164,669,40,13,2063


## Model-ready fold construction

This stage performs the same leakage checks again immediately before each training run, builds original-English contexts, injects only the matching CV-specific LLM rows, and appends the independent translated corpus to training only.

In [13]:
# Cell 13 — Build model-ready data for one CV run

def build_one_cv_data(cv_run):
    # Repeat deep audit immediately before data assembly.
    audit_cv_fold(cv_run, verbose=False)

    train_ids, val_ids, test_ids = load_cv_ids(cv_run)
    aug_df, aug_path = load_llm_augmentation(cv_run)

    train_original_df = en_target_df[en_target_df['conversation_id'].isin(train_ids)].copy().reset_index(drop=True)
    val_df = en_target_df[en_target_df['conversation_id'].isin(val_ids)].copy().reset_index(drop=True)
    test_df = en_target_df[en_target_df['conversation_id'].isin(test_ids)].copy().reset_index(drop=True)

    X_train_original, y_train_original, train_meta = build_context_dataset(
        train_original_df, en_turns_by_conv, return_meta=True
    )
    X_val, y_val, val_meta = build_context_dataset(
        val_df, en_turns_by_conv, return_meta=True
    )
    X_test, y_test, test_meta = build_context_dataset(
        test_df, en_turns_by_conv, return_meta=True
    )
    X_llm, y_llm, llm_meta = build_context_dataset(
        aug_df, en_turns_by_conv, return_meta=True
    )

    # Provenance-level key overlap checks (synthetic intentionally shares source TRAIN keys).
    train_conv = {cid for cid, _ in train_meta}
    val_conv = {cid for cid, _ in val_meta}
    test_conv = {cid for cid, _ in test_meta}
    llm_conv = {cid for cid, _ in llm_meta}
    assert train_conv.isdisjoint(val_conv)
    assert train_conv.isdisjoint(test_conv)
    assert val_conv.isdisjoint(test_conv)
    assert llm_conv.issubset(train_conv)
    assert llm_conv.isdisjoint(val_conv)
    assert llm_conv.isdisjoint(test_conv)

    # Translated data is training-only and namespaced.
    assert all(str(cid).startswith('CN_') for cid, _ in cn_meta)
    assert not any(str(cid).startswith('CN_') for cid, _ in val_meta + test_meta)

    X_train = np.concatenate([X_train_original, X_llm, X_cn_train])
    y_train = np.concatenate([y_train_original, y_llm, y_cn_train])

    # Diagnostic exact-context overlap only. Repeated generic language can naturally occur,
    # so we print rather than assert on exact text equality.
    norm = lambda x: re.sub(r'\\s+', ' ', str(x).strip().lower())
    train_texts = {norm(x) for x in X_train_original}
    val_texts = {norm(x) for x in X_val}
    test_texts = {norm(x) for x in X_test}
    print(f'CV{cv_run} exact ORIGINAL context overlaps: train∩val={len(train_texts & val_texts)}, train∩test={len(train_texts & test_texts)}, val∩test={len(val_texts & test_texts)}')
    print('Note: exact text repetition across different conversation IDs is not conversation leakage; membership remains ID-disjoint.')

    return {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val,
        'X_test': X_test,
        'y_test': y_test,
        'train_original_df': train_original_df,
        'val_df': val_df,
        'test_df': test_df,
        'aug_df': aug_df,
        'train_ids': train_ids,
        'val_ids': val_ids,
        'test_ids': test_ids,
        'test_meta': test_meta,
        'aug_path': aug_path,
    }

In [14]:
# Cell 14 — Smoke-inspect CV1 model-ready distributions before training
smoke_data = build_one_cv_data(1)

print('\nCV1 final model-ready sizes:')
print('Original English train:', len(smoke_data['train_original_df']))
print('LLM synthetic:', len(smoke_data['aug_df']))
print('Translated CN→EN:', len(cn_target_df))
print('FINAL train:', len(smoke_data['X_train']))
print('Validation:', len(smoke_data['X_val']))
print('Test:', len(smoke_data['X_test']))

print('\nCV1 FINAL TRAIN distribution:')
print(pd.Series(smoke_data['y_train']).value_counts().reindex(VALID_LABELS, fill_value=0))
print('\nCV1 VAL distribution:')
print(pd.Series(smoke_data['y_val']).value_counts().reindex(VALID_LABELS, fill_value=0))
print('\nCV1 TEST distribution:')
print(pd.Series(smoke_data['y_test']).value_counts().reindex(VALID_LABELS, fill_value=0))

CV1 exact ORIGINAL context overlaps: train∩val=16, train∩test=11, val∩test=10
Note: exact text repetition across different conversation IDs is not conversation leakage; membership remains ID-disjoint.

CV1 final model-ready sizes:
Original English train: 2030
LLM synthetic: 227
Translated CN→EN: 3056
FINAL train: 5313
Validation: 526
Test: 640

CV1 FINAL TRAIN distribution:
Chitchat      548
Interpret    2737
Reveal       1635
Revise        354
Inquire        39
Name: count, dtype: int64

CV1 VAL distribution:
Chitchat      42
Interpret    303
Reveal       164
Revise        15
Inquire        2
Name: count, dtype: int64

CV1 TEST distribution:
Chitchat      53
Interpret    363
Reveal       207
Revise        13
Inquire        4
Name: count, dtype: int64


## Training utilities

BERT-base uses the **same** learning rate (`2e-5`) and weight decay (`0.01`) as the 4-class experiment. Balanced class weights are computed from each fold's **final 5-class training data only**. Validation Macro-F1 selects the best in-memory model state; test data is never used for checkpoint selection.


In [17]:
# Cell 15 — Label encoding, datasets, metrics

def labels_to_ids(labels):
    return np.array([label2id[str(x)] for x in labels], dtype=np.int64)


def make_hf_datasets(tokenizer, data):
    def encode(texts):
        return tokenizer(
            list(texts),
            truncation=True,
            max_length=MAX_LEN,
        )

    y_train_ids = labels_to_ids(data['y_train'])
    y_val_ids = labels_to_ids(data['y_val'])
    y_test_ids = labels_to_ids(data['y_test'])

    train_ds = Dataset.from_dict({**encode(data['X_train']), 'labels': y_train_ids})
    val_ds = Dataset.from_dict({**encode(data['X_val']), 'labels': y_val_ids})
    test_ds = Dataset.from_dict({**encode(data['X_test']), 'labels': y_test_ids})

    return train_ds, val_ds, test_ds, y_train_ids, y_val_ids, y_test_ids


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds,  labels=np.arange(len(VALID_LABELS)),average='macro', zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    _, _, class_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=np.arange(len(VALID_LABELS)),
        average=None,
        zero_division=0,
    )

    out = {
        'accuracy': accuracy_score(labels, preds),
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    }
    for i, label in id2label.items():
        out[f'{label.lower()}_f1'] = float(class_f1[i])
    return out

In [18]:
# Cell 16 — Weighted Trainer and validation-only best-state callback

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        weight = self.class_weights.to(
        device=logits.device,
        dtype=logits.dtype,
        )

        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


class BestValidationStateCallback(TrainerCallback):
    def __init__(self, metric_name='eval_macro_f1'):
        self.metric_name = metric_name
        self.best_metric = -np.inf
        self.best_epoch = None
        self.best_step = None
        self.best_state_dict = None
        self.best_metrics = None

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or model is None:
            return control
        current = metrics.get(self.metric_name)
        if current is None:
            return control
        if float(current) > self.best_metric:
            self.best_metric = float(current)
            self.best_epoch = float(state.epoch) if state.epoch is not None else None
            self.best_step = int(state.global_step)
            self.best_state_dict = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            self.best_metrics = {
                k: float(v)
                for k, v in metrics.items()
                if isinstance(v, (int, float, np.integer, np.floating))
            }
        return control

In [19]:
# Cell 17 — Test evaluator

def evaluate_test(trainer, test_ds, y_test_ids, data, model_label, cv_run, output_dir):
    pred = trainer.predict(test_ds)
    y_pred_ids = np.argmax(pred.predictions, axis=1)

    y_true = [id2label[int(i)] for i in y_test_ids]
    y_pred = [id2label[int(i)] for i in y_pred_ids]

    print(f'\n=== {model_label} | CV{cv_run} | HELD-OUT TEST ===')
    report_text = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        digits=4,
        zero_division=0,
    )
    print(report_text)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report_dict).transpose().to_csv(
        os.path.join(output_dir, 'test_classification_report.csv')
    )

    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
    pd.DataFrame(cm, index=VALID_LABELS, columns=VALID_LABELS).to_csv(
        os.path.join(output_dir, 'test_confusion_matrix.csv')
    )

    pred_rows = []
    for (cid, ti), true_label, pred_label in zip(data['test_meta'], y_true, y_pred):
        pred_rows.append({
            'conversation_id': cid,
            'turn_index': ti,
            'true_label': true_label,
            'predicted_label': pred_label,
        })
    pd.DataFrame(pred_rows).to_csv(os.path.join(output_dir, 'test_predictions.csv'), index=False)

    macro_f1 = float(report_dict['macro avg']['f1-score'])
    weighted_f1 = float(report_dict['weighted avg']['f1-score'])
    accuracy = float(report_dict['accuracy'])

    result = {
        'model': model_label,
        'cv': cv_run,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'test_examples': len(y_true),
    }
    for label in VALID_LABELS:
        result[f'{label.lower()}_f1'] = float(report_dict[label]['f1-score'])
        result[f'{label.lower()}_support'] = int(report_dict[label]['support'])
    return result

In [20]:
# Cell 18 — Train one model on one frozen CV run

def train_one_model_one_cv(model_label, model_name, cv_run):
    print('\n' + '#' * 90)
    print(f'{model_label} | CV{cv_run}')
    print('#' * 90)

    set_seed(RANDOM_STATE)
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    # Repeat leakage audit immediately before this training run.
    audit_cv_fold(cv_run, verbose=True)
    data = build_one_cv_data(cv_run)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    train_ds, val_ds, test_ds, y_train_ids, y_val_ids, y_test_ids = make_hf_datasets(tokenizer, data)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Balanced class weights from FINAL TRAIN only.
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(len(VALID_LABELS)),
        y=y_train_ids,
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
    print('TRAIN-only class weights:', dict(zip(VALID_LABELS, class_weights)))

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(VALID_LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    safe_model = re.sub(r'[^A-Za-z0-9_.-]+', '_', model_label)
    output_dir = os.path.join(RESULTS_DIR, safe_model, f'cv{cv_run}')
    os.makedirs(output_dir, exist_ok=True)

    best_cb = BestValidationStateCallback(metric_name='eval_macro_f1')

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy='epoch',
        save_strategy='no',
        logging_strategy='epoch',
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        load_best_model_at_end=False,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[best_cb],
    )

    trainer.train()

    assert best_cb.best_state_dict is not None, 'No best validation model state captured.'
    trainer.model.load_state_dict(best_cb.best_state_dict, strict=True)

    print('\nBest validation epoch:', best_cb.best_epoch)
    print('Best validation Macro-F1:', best_cb.best_metric)

    best_model_dir = os.path.join(output_dir, 'best_validation_model')
    os.makedirs(best_model_dir, exist_ok=True)
    trainer.model.save_pretrained(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)

    result = evaluate_test(
        trainer, test_ds, y_test_ids, data, model_label, cv_run, output_dir
    )
    result['best_val_macro_f1'] = best_cb.best_metric
    result['best_val_epoch'] = best_cb.best_epoch
    result['learning_rate'] = LEARNING_RATE
    result['weight_decay'] = WEIGHT_DECAY
    result['seed'] = RANDOM_STATE
    result['final_train_examples'] = len(data['X_train'])
    result['val_examples'] = len(data['X_val'])

    pd.DataFrame([result]).to_csv(os.path.join(output_dir, 'test_metrics.csv'), index=False)

    del trainer, model, tokenizer, train_ds, val_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Run configuration

By default, BERT-base is enabled and `RUN_FOLDS = [1, 2, 3, 4, 5]`.

For a smoke test first, change:

```python
RUN_FOLDS = [1]
```

Then restore `[1, 2, 3, 4, 5]` for the complete 5-CV experiment.


In [21]:
# Cell 19 — Execute enabled models and requested folds
all_results = []

for model_label, cfg in MODEL_CONFIGS.items():
    if not cfg['enabled']:
        print(f'SKIP {model_label} (disabled)')
        continue

    for cv_run in RUN_FOLDS:
        result = train_one_model_one_cv(
            model_label=model_label,
            model_name=cfg['model_name'],
            cv_run=cv_run,
        )
        all_results.append(result)

        # Save after every completed run.
        pd.DataFrame(all_results).to_csv(
            os.path.join(RESULTS_DIR, 'running_model_cv_results.csv'),
            index=False,
        )

results_df = pd.DataFrame(all_results)
display(results_df)

SKIP BERT-base (disabled)

##########################################################################################
DistilBERT | CV1
##########################################################################################
CV1: PASS — no conversation-ID or target-turn leakage; LLM provenance verified.
  Train/Val/Test conversations: 528 132 166
  Original target rows: 2030 526 640
  LLM distribution: {'Chitchat': 100, 'Revise': 100, 'Inquire': 27}
  LLM rows / source conversations: 227 76
  LLM global exact-text duplicates (audit only): 0
  LLM source ∩ Validation: 0
  LLM source ∩ Test: 0
CV1 exact ORIGINAL context overlaps: train∩val=16, train∩test=11, val∩test=10
Note: exact text repetition across different conversation IDs is not conversation leakage; membership remains ID-disjoint.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Chitchat F1,Interpret F1,Reveal F1,Revise F1,Inquire F1
1,1.062200,0.728837,0.860902,0.552020,0.466054,0.497891,0.860442,0.756757,0.901615,0.831081,0.000000,0.000000
2,0.811400,0.639289,0.825188,0.541113,0.452476,0.487649,0.841774,0.729730,0.870229,0.838284,0.000000,0.000000
3,0.685700,0.635378,0.860902,0.530720,0.473395,0.496381,0.861545,0.736842,0.897710,0.847352,0.000000,0.000000
4,0.567900,0.705496,0.853383,0.531465,0.465357,0.491623,0.857556,0.720000,0.896341,0.841772,0.000000,0.000000
5,0.499300,0.689287,0.851504,0.514667,0.474606,0.492153,0.856220,0.725000,0.892691,0.843077,0.000000,0.000000



Best validation epoch: 1.0
Best validation Macro-F1: 0.4978906218994325



=== MiniLM | CV5 | HELD-OUT TEST ===
              precision    recall  f1-score   support

    Chitchat     0.8611    0.5741    0.6889        54
   Interpret     0.8071    0.9449    0.8706       363
      Reveal     0.9423    0.7136    0.8122       206
      Revise     0.0000    0.0000    0.0000        14
     Inquire     0.0000    0.0000    0.0000         3

    accuracy                         0.8141       640
   macro avg     0.5221    0.4465    0.4743       640
weighted avg     0.8337    0.8141    0.8133       640



,model,cv,accuracy,macro_f1,weighted_f1,test_examples,chitchat_f1,chitchat_support,interpret_f1,interpret_support,reveal_f1,reveal_support,revise_f1,revise_support,inquire_f1,inquire_support,best_val_macro_f1,best_val_epoch,learning_rate,weight_decay,seed,final_train_examples,val_examples
0,DistilBERT,1,0.810937,0.463361,0.803756,640,0.643678,53,0.858667,363,0.814458,207,0.000000,13,0.000000,4,0.560426,3.0,0.00002,0.01,42,5313,526
1,DistilBERT,2,0.852895,0.498597,0.843233,639,0.537634,54,0.910786,362,0.862745,206,0.181818,13,0.000000,4,0.514081,4.0,0.00002,0.01,42,5327,511
2,DistilBERT,3,0.843260,0.600570,0.839475,638,0.679612,53,0.887686,362,0.852217,206,0.083333,14,0.500000,3,0.515047,4.0,0.00002,0.01,42,5328,516
3,DistilBERT,4,0.832551,0.492458,0.829954,639,0.620690,54,0.888604,363,0.838710,206,0.114286,13,0.000000,3,0.552869,1.0,0.00002,0.01,42,5351,494
4,DistilBERT,5,0.829688,0.518358,0.829765,640,0.627451,54,0.877005,363,0.852041,206,0.235294,14,0.000000,3,0.499734,5.0,0.00002,0.01,42,5315,532
5,RoBERTa,1,0.800000,0.450727,0.787810,640,0.617284,53,0.851010,363,0.785340,207,0.000000,13,0.000000,4,0.517539,1.0,0.00002,0.01,42,5313,526
6,RoBERTa,2,0.823161,0.626634,0.824824,639,0.554217,54,0.885870,362,0.831296,206,0.195122,13,0.666667,4,0.505187,3.0,0.00002,0.01,42,5327,511
7,RoBERTa,3,0.805643,0.616451,0.811066,638,0.639175,53,0.861373,362,0.817481,206,0.097561,14,0.666667,3,0.525752,2.0,0.00002,0.01,42,5328,516
8,RoBERTa,4,0.841941,0.502703,0.836256,639,0.636364,54,0.897338,363,0.836957,206,0.142857,13,0.000000,3,0.537734,2.0,0.00002,0.01,42,5351,494
9,RoBERTa,5,0.820312,0.490595,0.810872,640,0.659794,54,0.869565,363,0.805970,206,0.117647,14,0.000000,3,0.537657,1.0,0.00002,0.01,42,5315,532


In [22]:
# Cell 20 — Completed-run summary (mean ± SD when multiple folds are available)

if len(results_df) == 0:
    print('No model runs completed.')

else:
    metrics = [
        'accuracy',
        'macro_f1',
        'weighted_f1',
        'chitchat_f1',
        'interpret_f1',
        'reveal_f1',
        'revise_f1',
        'inquire_f1',
    ]

    summary_rows = []

    for model_name, group in results_df.groupby('model'):

        row = {
            'Model': model_name,
            'Completed_folds': len(group)
        }

        for m in metrics:

            mean = group[m].mean()
            std = (
                group[m].std(ddof=1)
                if len(group) > 1
                else np.nan
            )

            row[f'{m}_mean'] = mean
            row[f'{m}_std'] = std

            row[f'{m}_mean±sd'] = (
                f'{mean:.4f} ± {std:.4f}'
                if len(group) > 1
                else f'{mean:.4f} (1 fold)'
            )

        summary_rows.append(row)

    model_cv_summary = pd.DataFrame(
        summary_rows
    )

    display(model_cv_summary)

    model_cv_summary.to_csv(
        os.path.join(
            RESULTS_DIR,
            'model_5cv_summary.csv'
        ),
        index=False,
    )


,Model,Completed_folds,accuracy_mean,accuracy_std,accuracy_mean±sd,macro_f1_mean,macro_f1_std,macro_f1_mean±sd,weighted_f1_mean,weighted_f1_std,weighted_f1_mean±sd,chitchat_f1_mean,chitchat_f1_std,chitchat_f1_mean±sd,interpret_f1_mean,interpret_f1_std,interpret_f1_mean±sd,reveal_f1_mean,reveal_f1_std,reveal_f1_mean±sd,revise_f1_mean,revise_f1_std,revise_f1_mean±sd,inquire_f1_mean,inquire_f1_std,inquire_f1_mean±sd
0,DistilBERT,5,0.833866,0.015774,0.8339 ± 0.0158,0.514669,0.051906,0.5147 ± 0.0519,0.829237,0.015417,0.8292 ± 0.0154,0.621813,0.052292,0.6218 ± 0.0523,0.884550,0.018983,0.8845 ± 0.0190,0.844034,0.018603,0.8440 ± 0.0186,0.122946,0.090620,0.1229 ± 0.0906,0.100000,0.223607,0.1000 ± 0.2236
1,MiniLM,5,0.823538,0.014720,0.8235 ± 0.0147,0.480719,0.014346,0.4807 ± 0.0143,0.817809,0.013811,0.8178 ± 0.0138,0.630581,0.063994,0.6306 ± 0.0640,0.876050,0.011116,0.8760 ± 0.0111,0.826214,0.025811,0.8262 ± 0.0258,0.070749,0.049865,0.0707 ± 0.0499,0.000000,0.000000,0.0000 ± 0.0000
2,RoBERTa,5,0.818211,0.016444,0.8182 ± 0.0164,0.537422,0.079244,0.5374 ± 0.0792,0.814166,0.018143,0.8142 ± 0.0181,0.621367,0.040449,0.6214 ± 0.0404,0.873031,0.018631,0.8730 ± 0.0186,0.815409,0.020701,0.8154 ± 0.0207,0.110637,0.071827,0.1106 ± 0.0718,0.266667,0.365148,0.2667 ± 0.3651


In [23]:
# Cell 21 — End-of-notebook fold data summary
# This is independent of which models/folds were trained in the current runtime.

cols = [
    'CV',
    'Train_conversations',
    'Val_conversations',
    'Test_conversations',

    'Train_original_targets',
    'LLM_rows',
    'LLM_global_exact_text_duplicates',
    'Translated_targets',
    'Final_train_examples',

    'Val_targets',
    'Test_targets',

    'TrainOriginal_Chitchat',
    'TrainOriginal_Interpret',
    'TrainOriginal_Reveal',
    'TrainOriginal_Revise',
    'TrainOriginal_Inquire',

    'LLM_Chitchat',
    'LLM_Interpret',
    'LLM_Reveal',
    'LLM_Revise',
    'LLM_Inquire',

    'FinalTrain_Chitchat',
    'FinalTrain_Interpret',
    'FinalTrain_Reveal',
    'FinalTrain_Revise',
    'FinalTrain_Inquire',

    'Val_Chitchat',
    'Val_Interpret',
    'Val_Reveal',
    'Val_Revise',
    'Val_Inquire',

    'Test_Chitchat',
    'Test_Interpret',
    'Test_Reveal',
    'Test_Revise',
    'Test_Inquire',
]

print('FINAL DATA AUDIT SUMMARY — ALL 5 CV RUNS')
display(fold_data_summary[cols])

print('\nOuter TEST fold distribution only:')

test_only = (
    original_split_distribution[
        original_split_distribution['Split'] == 'Test'
    ]
    .reset_index(drop=True)
)

display(test_only)

print('\nPASS criteria already asserted above:')
print('- Train/Val/Test conversation IDs disjoint inside every CV')
print('- Each outer test fold pairwise disjoint')
print('- Every one of the 826 eligible English conversations is test exactly once across 5 CV')
print('- Every LLM synthetic source belongs to that CV TRAIN only')
print('- Every LLM synthetic row maps to a real original TRAIN source turn with the SAME label')
print('- Inquire synthetic rows are verified against original Inquire source turns')
print('- Validation and test contain only original, unaugmented English target turns')
print('- Translated corpus isolated with CN_ namespace and used only for training')
print('- Class weights are computed from FINAL 5-class training data only')
print('- Best state is chosen only from validation Macro-F1')
print('- Test is evaluated only after restoring the best validation state')


FINAL DATA AUDIT SUMMARY — ALL 5 CV RUNS


,CV,Train_conversations,Val_conversations,Test_conversations,Train_original_targets,LLM_rows,LLM_global_exact_text_duplicates,Translated_targets,Final_train_examples,Val_targets,Test_targets,TrainOriginal_Chitchat,TrainOriginal_Interpret,TrainOriginal_Reveal,TrainOriginal_Revise,TrainOriginal_Inquire,LLM_Chitchat,LLM_Interpret,LLM_Reveal,LLM_Revise,LLM_Inquire,FinalTrain_Chitchat,FinalTrain_Interpret,FinalTrain_Reveal,FinalTrain_Revise,FinalTrain_Inquire,Val_Chitchat,Val_Interpret,Val_Reveal,Val_Revise,Val_Inquire,Test_Chitchat,Test_Interpret,Test_Reveal,Test_Revise,Test_Inquire
0,1,528,132,166,2030,227,0,3056,5313,526,640,173,1147,660,39,11,100,0,0,100,27,548,2737,1635,354,39,42,303,164,15,2,53,363,207,13,4
1,2,529,132,165,2046,225,62,3056,5327,511,639,174,1163,653,46,10,100,0,0,100,25,549,2753,1628,361,36,40,288,172,8,3,54,362,206,13,4
2,3,529,132,165,2042,230,59,3056,5328,516,638,166,1145,674,45,12,100,0,0,100,30,541,2735,1649,360,43,49,306,151,8,2,53,362,206,14,3
3,4,529,132,165,2063,232,52,3056,5351,494,639,177,1164,669,40,13,100,0,0,100,32,552,2754,1644,355,46,37,286,156,14,1,54,363,206,13,3
4,5,529,132,165,2024,235,45,3056,5315,532,640,169,1134,658,49,14,100,0,0,100,35,544,2724,1633,364,50,45,316,167,4,0,54,363,206,14,3



Outer TEST fold distribution only:


,CV,Split,Conversations,Chitchat,Interpret,Reveal,Revise,Inquire,Total_examples
0,CV1,Test,166,53,363,207,13,4,640
1,CV2,Test,165,54,362,206,13,4,639
2,CV3,Test,165,53,362,206,14,3,638
3,CV4,Test,165,54,363,206,13,3,639
4,CV5,Test,165,54,363,206,14,3,640



PASS criteria already asserted above:
- Train/Val/Test conversation IDs disjoint inside every CV
- Each outer test fold pairwise disjoint
- Every one of the 826 eligible English conversations is test exactly once across 5 CV
- Every LLM synthetic source belongs to that CV TRAIN only
- Every LLM synthetic row maps to a real original TRAIN source turn with the SAME label
- Inquire synthetic rows are verified against original Inquire source turns
- Validation and test contain only original, unaugmented English target turns
- Translated corpus isolated with CN_ namespace and used only for training
- Class weights are computed from FINAL 5-class training data only
- Best state is chosen only from validation Macro-F1
- Test is evaluated only after restoring the best validation state
